# Part 1 — Can we trust the numbers? A data-quality story in ten acts

**Context.** The squad's notebook concludes that a 14-day trailing average plus a fixed-threshold
rule "looks like it's working". Before judging any model, this notebook asks a more basic
question: *is the data going into it trustworthy?*

**Short answer: no — but almost every problem can be fixed exactly, not statistically.** The data
carries its own answer key, the *ledger identity*:

> `balance_t = balance_(t-1) + inflow_t − outflow_t`

It holds to the cent on every clean row. That single fact lets us pick the right duplicate, fill
missing values, repair corrupted readings — and, just as important, tell *which* readings we
**cannot** vouch for, instead of quietly smoothing them away.

**How each act reads:** *what we saw → why it matters for a cash decision → what we did → proof.*

| Act | Problem | Resolution |
|---|---|---|
| 1 | Squad's date parsing deletes 17.6% of rows | Parse by separator rule (zero exceptions) |
| 2 | One currency written up to three ways | Normalize; `accounts.csv` is the source of truth |
| 3 | No way to tell good balances from bad | Ledger identity + reconciliation rules |
| 4 | 48 duplicated (account, day) pairs, 44 with conflicting balances | Keep the row that reconciles — never average |
| 5 | ~4% of balances / flows missing | Recover exactly from the identity; backtested |
| 6 | Negative balances, 10× spikes, unexplained 3-day drops | Repair exact signatures; quarantine the rest |
| 7 | — | Clean dataset, audited and saved |
| 8 | 59 "statistical outliers" | Separate proven errors from real variability |
| 9 | Is there structure to forecast? | Weekly rhythm in flows, none in the balance level |
| 10 | Does any of this change the squad's answer? | Yes for the forecast series; no for the final-day recommendation |

In [1]:
import re
import warnings

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

accounts = pd.read_csv("accounts.csv")
raw = pd.read_csv(
    "account_balances_daily_RAW.csv",
    dtype={"account_id": str, "date": str, "balance": float, "currency": str},
)
transfers = pd.read_csv("transfers_log_RAW.csv", parse_dates=["date_requested", "date_settled"])

SQUAD_THRESHOLD = {"ACC-001": 1_000_000_000, "ACC-002": 100_000, "ACC-004": 40_000_000, "ACC-006": 100_000}
ACCOUNT_COLORS = {
    "ACC-001": "#4C72B0", "ACC-002": "#DD8452", "ACC-003": "#55A868",
    "ACC-004": "#C44E52", "ACC-005": "#8172B3", "ACC-006": "#937860",
}

print(f"raw balance rows: {len(raw):,} | accounts: {len(accounts)} | transfers: {len(transfers)}")
accounts

raw balance rows: 1,668 | accounts: 6 | transfers: 145


,account_id,bank_name,currency,account_type
0,ACC-001,Banco Aurora,COP,operational
1,ACC-002,Banco Aurora,USD,operational
2,ACC-003,Banco Sureste,COP,reserve
3,ACC-004,Banco Azteca+,MXN,operational
4,ACC-005,Banco Azteca+,MXN,reserve
5,ACC-006,Banco Norte,USD,reserve


## Act 1 — The squad's pipeline silently deletes 17.6% of the data

**What we saw.** The squad parses `date` with `pd.to_datetime(errors="coerce")` and drops whatever
fails, noting "a few dates don't parse". Let's count how many that is.

In [2]:
def classify_date_format(s):
    if pd.isna(s):
        return "missing"
    if re.fullmatch(r"\d{4}-\d{2}-\d{2}", s):
        return "iso_YYYY-MM-DD"
    m = re.fullmatch(r"(\d{2})/(\d{2})/(\d{4})", s)
    if m:
        a, b = int(m.group(1)), int(m.group(2))
        if a > 12 >= b: return "slash_DD/MM/YYYY"
        if b > 12 >= a: return "slash_MM/DD/YYYY"
        return "slash_ambiguous (both<=12)"
    m = re.fullmatch(r"(\d{2})-(\d{2})-(\d{4})", s)
    if m:
        a, b = int(m.group(1)), int(m.group(2))
        if a > 12 >= b: return "dash2_DD-MM-YYYY"
        if b > 12 >= a: return "dash2_MM-DD-YYYY"
        return "dash2_ambiguous (both<=12)"
    return "other/unrecognized"

raw["date_format"] = raw["date"].apply(classify_date_format)

naive_dates = pd.to_datetime(raw["date"], errors="coerce")
squad_rows = int(naive_dates.notna().sum())
dropped = len(raw) - squad_rows
print(f"Rows in the raw file:             {len(raw):,}")
print(f"Rows the squad's pipeline keeps:  {squad_rows:,}")
print(f"Rows silently dropped:            {dropped:,}  ({dropped / len(raw):.1%})")
print(f"Survivors that are ISO-formatted: {(raw.loc[naive_dates.notna(), 'date_format'] == 'iso_YYYY-MM-DD').mean():.0%}")
raw["date_format"].value_counts().to_frame("rows")

Rows in the raw file:             1,668
Rows the squad's pipeline keeps:  1,375
Rows silently dropped:            293  (17.6%)
Survivors that are ISO-formatted: 100%


,rows
date_format,
iso_YYYY-MM-DD,1375
slash_DD/MM/YYYY,106
dash2_MM-DD-YYYY,78
slash_ambiguous (both<=12),64
dash2_ambiguous (both<=12),45


It isn't "a few": **every single non-ISO date is discarded** — not because the data is corrupt, but
because pandas was never told the other layouts exist. The formats look randomly scattered
across accounts and currencies, so there is no per-bank convention to lean on. There *is* a
perfect pattern by separator, though:

In [3]:
exceptions = {
    "'/' dates that are month-first (MM/DD)": int((raw["date_format"] == "slash_MM/DD/YYYY").sum()),
    "'-' dates that are day-first (DD-MM)":   int((raw["date_format"] == "dash2_DD-MM-YYYY").sum()),
}
n_slash = raw["date_format"].str.startswith("slash").sum()
n_dash = raw["date_format"].str.startswith("dash2").sum()
print(f"'/' rows: {n_slash} | '-' (2-2-4) rows: {n_dash}")
print("Exceptions to 'slash = day-first, dash = month-first':", exceptions)


def parse_date_safely(s):
    if pd.isna(s):
        return pd.NaT
    if re.fullmatch(r"\d{4}-\d{2}-\d{2}", s):
        return pd.to_datetime(s, format="%Y-%m-%d")
    if re.fullmatch(r"\d{2}/\d{2}/\d{4}", s):
        return pd.to_datetime(s, format="%d/%m/%Y")
    if re.fullmatch(r"\d{2}-\d{2}-\d{4}", s):
        return pd.to_datetime(s, format="%m-%d-%Y")
    return pd.NaT


raw["date_parsed"] = raw["date"].apply(parse_date_safely)
print(f"\nRows recovered: {raw['date_parsed'].notna().sum():,} / {len(raw):,}")

coverage = raw.groupby("account_id")["date_parsed"].agg(first="min", last="max", rows="size", distinct_days="nunique")
coverage["calendar_days"] = (coverage["last"] - coverage["first"]).dt.days + 1
coverage["duplicate_rows"] = coverage["rows"] - coverage["distinct_days"]
coverage

'/' rows: 170 | '-' (2-2-4) rows: 123
Exceptions to 'slash = day-first, dash = month-first': {"'/' dates that are month-first (MM/DD)": 0, "'-' dates that are day-first (DD-MM)": 0}

Rows recovered: 1,668 / 1,668


,first,last,rows,distinct_days,calendar_days,duplicate_rows
account_id,,,,,,
ACC-001,2025-01-01,2025-09-27,277,270,270,7
ACC-002,2025-01-01,2025-09-27,284,270,270,14
ACC-003,2025-01-01,2025-09-27,279,270,270,9
ACC-004,2025-01-01,2025-09-27,278,270,270,8
ACC-005,2025-01-01,2025-09-27,276,270,270,6
ACC-006,2025-01-01,2025-09-27,274,270,270,4


Zero exceptions across all 1,668 rows, so the rule also settles the "ambiguous" dates (both parts
≤ 12). After the fix every account has exactly one row per calendar day for 270 days
(`distinct_days == calendar_days`), plus 4–14 surplus rows per account that are duplicates (48 in
total, Act 4). A concrete example of what the squad's pipeline was hiding:

In [4]:
raw.loc[raw["date"] == "24/06/2025", ["date", "account_id", "balance", "inflow", "outflow"]]

,date,account_id,balance,inflow,outflow
960,24/06/2025,ACC-003,"-2,890,859,520.02","105,959,244.11","114,391,526.36"


That `24/06/2025` row (a large negative balance on a reserve account) is invalid as month-24, so
the squad's `errors="coerce"` turned it into `NaT` and `dropna()` removed it. Nobody was told.

**Why it matters for a cash decision.** A forecast built on a series with 17.6% of its days
missing — unevenly, so windows silently stretch over 2, 3 or 4 calendar days — is a forecast of a
different series than the one treasury actually runs.

**Resolved:** `date_parsed` recovers 100% of rows with a documented, testable rule.

## Act 2 — One currency, up to nine spellings

**What we saw.** `currency` holds `COP`, `cop`, `Colombian Peso`, … — 9 labels for 3 currencies.
Any `groupby("currency")` would silently split one currency into several buckets.

In [5]:
CURRENCY_MAP = {
    "cop": "COP", "colombian peso": "COP",
    "usd": "USD", "us dollar": "USD",
    "mxn": "MXN", "mexican peso": "MXN",
}
raw["currency_label"] = raw["currency"]
raw["currency_norm"] = raw["currency_label"].str.strip().str.lower().map(CURRENCY_MAP)
assert raw["currency_norm"].notna().all(), "unmapped currency label"

check = raw.merge(accounts[["account_id", "currency"]].rename(columns={"currency": "account_currency"}), on="account_id")
print("Distinct raw labels:", raw["currency_label"].nunique(), "→ normalized:", raw["currency_norm"].nunique())
print("Rows whose normalized label disagrees with accounts.csv:", int((check["currency_norm"] != check["account_currency"]).sum()))
pd.crosstab(raw["account_id"], raw["currency_label"])

Distinct raw labels: 9 → normalized: 3
Rows whose normalized label disagrees with accounts.csv: 0


currency_label,COP,Colombian Peso,MXN,Mexican Peso,US Dollar,USD,cop,mxn,usd
account_id,,,,,,,,,
ACC-001,246,17,0,0,0,0,14,0,0
ACC-002,0,0,0,0,14,252,0,0,18
ACC-003,250,13,0,0,0,0,16,0,0
ACC-004,0,0,238,18,0,0,0,22,0
ACC-005,0,0,244,17,0,0,0,15,0
ACC-006,0,0,0,0,8,251,0,0,15


Each account only ever uses spellings of its own currency — **no row is in the wrong currency**.
So this is a labelling problem, not a data-integrity one, and `accounts.csv` is the source of
truth: we assign each account its catalogue currency and never convert (every series stays in
its native unit; no FX is needed anywhere in this notebook).

**Why it matters.** Currency is what keeps a COP billion from being compared to a USD hundred
thousand — which is exactly the mistake the squad's donor-selection rule makes later (Act 10).

**Resolved:** `currency` := `accounts.csv` currency, zero conflicts.

In [6]:
raw["currency"] = raw["account_id"].map(accounts.set_index("account_id")["currency"])
raw[["account_id", "currency"]].drop_duplicates().sort_values("account_id").reset_index(drop=True)

,account_id,currency
0,ACC-001,COP
1,ACC-002,USD
2,ACC-003,COP
3,ACC-004,MXN
4,ACC-005,MXN
5,ACC-006,USD


## Act 3 — The ledger never lies

**What we saw.** With dates fixed, the series still looks wrong in places (10× spikes, negative
balances). But *which* readings are wrong? "Looks odd" is a statistical opinion; we want proof.

**The key.** Cash only moves through `inflow` and `outflow`, so consecutive balances must satisfy
`balance_t = balance_(t-1) + inflow_t − outflow_t`. Let's test it, naively, on the date-fixed data
(duplicates averaged, nothing else touched):

In [7]:
naive_daily = (
    raw.groupby(["account_id", "date_parsed"], as_index=False)
    .agg(balance=("balance", "mean"), inflow=("inflow", "mean"), outflow=("outflow", "mean"))
    .sort_values(["account_id", "date_parsed"])
)
naive_daily["expected"] = (
    naive_daily.groupby("account_id")["balance"].shift(1) + naive_daily["inflow"] - naive_daily["outflow"]
)
naive_daily["residual"] = naive_daily["balance"] - naive_daily["expected"]

evaluable = naive_daily.dropna(subset=["residual"])
holds = evaluable["residual"].abs() <= 1.0
print(f"Day-pairs where the identity can be tested: {len(evaluable):,}")
print(f"  hold within 1 currency unit: {holds.sum():,} ({holds.mean():.1%})")
print(f"  break:                       {(~holds).sum():,} ({(~holds).mean():.1%})")
print(f"Median |residual| on the ones that hold: {evaluable.loc[holds, 'residual'].abs().median():.2e}")
print(f"Median |residual| on the ones that break: {evaluable.loc[~holds, 'residual'].abs().median():,.0f}")

Day-pairs where the identity can be tested: 1,371
  hold within 1 currency unit: 1,265 (92.3%)
  break:                       106 (7.7%)
Median |residual| on the ones that hold: 2.91e-11
Median |residual| on the ones that break: 57,473


The identity holds to rounding error on the vast majority of rows and fails *by a lot* on the rest
— a clean separation. That makes it a reliable detector. The reconciliation rules below turn it
into a repair procedure, in this order:

1. **Verify** — a balance is *trusted* if it satisfies the identity against a neighbouring day (either side) within 1 currency unit. Matching to that precision by chance is effectively impossible.
2. **Choose** — if a day has two conflicting readings, the one that reconciles wins.
3. **Fill** — if a day has no balance, compute it from the nearest trusted balance and that day's flows (forward, or backward if only the next day is trusted).
4. **Recover flows** — if one flow is missing but both balances are trusted, solve for it.
5. **Repair** — if a reading fails the identity but equals exactly −1× or 10× what the ledger expects, it is a sign flip / decimal shift: replace it with the ledger value.
6. **Accept with a flag** — if a reading can't be checked (adjacent flows missing) but is within 25% of the nearest trusted balance, keep it as `observed_unverified`. (Reconciled day-to-day moves never exceed ~10%; the anomalies we'll meet drop 65–85%.)
7. **Quarantine** — anything else that contradicts the flows is *not* fixed and *not* guessed: its balance is set to missing and reported as an unreconciled episode.

In [8]:
TOL = 1.0                       # currency units; every clean row reconciles to <= 0.01
PLAUSIBLE_REL_CHANGE = 0.25     # for readings we cannot verify (rule 6)
SIGNATURES = {"sign_flip": -1.0, "decimal_shift_x10": 10.0}


def match_signature(observed, expected):
    if not expected:
        return None
    ratio = observed / expected
    for name, k in SIGNATURES.items():
        if abs(ratio - k) <= 1e-3 * abs(k):
            return name
    return None


def reconcile_account(cands, inflow, outflow, tol=TOL):
    """cands: per-day list of reported balances (0, 1 or 2 values). Returns per-day arrays."""
    n = len(cands)
    bal = np.full(n, np.nan)
    status = np.array([""] * n, dtype=object)
    flow_status = np.array([""] * n, dtype=object)
    contradicts = np.zeros(n, dtype=bool)
    inflow, outflow = inflow.copy(), outflow.copy()
    known = lambda x: not np.isnan(x)

    def expected(t):
        if t > 0 and known(bal[t - 1]) and known(inflow[t]) and known(outflow[t]):
            return bal[t - 1] + inflow[t] - outflow[t], "forward"
        if t < n - 1 and known(bal[t + 1]) and known(inflow[t + 1]) and known(outflow[t + 1]):
            return bal[t + 1] - inflow[t + 1] + outflow[t + 1], "backward"
        return None, None

    def verify_against_reported_neighbours():
        for t in range(n):
            if known(bal[t]):
                continue
            for c in cands[t]:
                ok = False
                if t > 0 and known(inflow[t]) and known(outflow[t]):
                    ok |= any(abs(c - (p + inflow[t] - outflow[t])) <= tol for p in cands[t - 1])
                if t < n - 1 and known(inflow[t + 1]) and known(outflow[t + 1]):
                    ok |= any(abs(nb - (c + inflow[t + 1] - outflow[t + 1])) <= tol for nb in cands[t + 1])
                if ok:
                    bal[t], status[t] = c, "observed_reconciled"
                    break

    def propagate():
        for _ in range(20):
            changed = False
            verify_against_reported_neighbours()
            for t in range(n):
                if known(bal[t]):
                    continue
                exp, how = expected(t)
                if exp is None:
                    continue
                if not cands[t]:
                    bal[t], status[t], changed = exp, f"imputed_{how}", True
                elif any(abs(c - exp) <= tol for c in cands[t]):
                    bal[t] = min(cands[t], key=lambda c: abs(c - exp))
                    status[t], changed = "observed_reconciled", True
                else:
                    sig = match_signature(cands[t][0], exp)
                    if sig:
                        bal[t], status[t], changed = exp, f"repaired_{sig}", True
                    else:
                        contradicts[t] = True
            for t in range(1, n):
                if known(bal[t]) and known(bal[t - 1]):
                    net = bal[t] - bal[t - 1]
                    if not known(inflow[t]) and known(outflow[t]) and net + outflow[t] >= -tol:
                        inflow[t], flow_status[t], changed = max(net + outflow[t], 0.0), "inflow_recovered", True
                    elif not known(outflow[t]) and known(inflow[t]) and inflow[t] - net >= -tol:
                        outflow[t], flow_status[t], changed = max(inflow[t] - net, 0.0), "outflow_recovered", True
            if not changed:
                break

    propagate()
    for _ in range(5):
        accepted = False
        for i in range(n):
            if known(bal[i]) or not cands[i] or contradicts[i]:
                continue
            nearest = [j for j in sorted(range(n), key=lambda j: abs(j - i)) if known(bal[j])]
            if nearest and bal[nearest[0]] and abs(cands[i][0] / bal[nearest[0]] - 1) <= PLAUSIBLE_REL_CHANGE:
                bal[i], status[i], accepted = cands[i][0], "observed_unverified", True
        if not accepted:
            break
        propagate()
    for i in range(n):
        if not known(bal[i]):
            status[i] = "unreconciled_episode" if cands[i] else "unresolved_missing"
    return bal, status, inflow, outflow, flow_status


def reconcile_ledger(df, tol=TOL):
    out = []
    for acc, g in df.groupby("account_id"):
        days = pd.date_range(g["date_parsed"].min(), g["date_parsed"].max(), freq="D")
        by_day = g.groupby("date_parsed")
        cands = by_day["balance"].apply(lambda s: s.dropna().tolist()).reindex(days)
        cands = [c if isinstance(c, list) else [] for c in cands]
        flows = by_day[["inflow", "outflow"]].first().reindex(days)
        bal, status, infl, outf, fstat = reconcile_account(
            cands, flows["inflow"].to_numpy(float), flows["outflow"].to_numpy(float), tol
        )
        out.append(pd.DataFrame({
            "account_id": acc, "date": days,
            "balance_clean": bal, "balance_status": status,
            "inflow_clean": infl, "outflow_clean": outf, "flow_status": fstat,
            "inflow_raw": flows["inflow"].to_numpy(float), "outflow_raw": flows["outflow"].to_numpy(float),
            "n_rows": by_day.size().reindex(days).fillna(0).astype(int).to_numpy(),
            "n_candidates": [len(c) for c in cands],
            "cand_1": [c[0] if len(c) > 0 else np.nan for c in cands],
            "cand_2": [c[1] if len(c) > 1 else np.nan for c in cands],
        }))
    return pd.concat(out, ignore_index=True)


ledger = reconcile_ledger(raw)
print(f"account-days reconciled: {len(ledger):,}  ({ledger['account_id'].nunique()} accounts x {ledger.groupby('account_id').size().iloc[0]} days)")
ledger["balance_status"].value_counts().to_frame("account_days")

account-days reconciled: 1,620  (6 accounts x 270 days)


,account_days
balance_status,
observed_reconciled,1520
imputed_forward,58
unreconciled_episode,18
observed_unverified,11
imputed_backward,6
repaired_sign_flip,4
repaired_decimal_shift_x10,3


Every account-day now has exactly one verdict. Almost everything is either *observed and
reconciled* or *imputed exactly*; a handful were repaired; a handful are unverifiable; and
18 days could not be reconciled at all. The next acts open each bucket.

## Act 4 — Duplicates: pick the row that reconciles, never average

**What we saw.** 48 (account, day) pairs appear twice — 44 with two different balances, 4 with both
blank. Both rows carry the *same* date
string, currency label, `inflow` and `outflow` — only `balance` differs, by a small amount. That
looks like a re-load with a slightly different snapshot, not two real events. Which one is right?

In [9]:
dups = ledger[ledger["n_rows"] == 2].copy()
both = dups[dups["n_candidates"] == 2].copy()
both["kept"] = both["balance_clean"]
is_c1 = (both["cand_1"] - both["kept"]).abs() <= TOL
is_c2 = (both["cand_2"] - both["kept"]).abs() <= TOL
both["kept_is_a_candidate"] = is_c1 | is_c2
both["dropped"] = np.where(is_c1, both["cand_2"], both["cand_1"])
both["dropped_off_by"] = both["dropped"] - both["kept"]
both["dropped_off_by_pct"] = both["dropped_off_by"] / both["kept"] * 100
both["mean_rule_error"] = ((both["cand_1"] + both["cand_2"]) / 2 - both["kept"]).abs()

print(f"Duplicated (account, day) pairs: {len(dups)}")
print(f"  both rows carry a balance:      {(dups['n_candidates'] == 2).sum()}")
print(f"  only one row carries a balance: {(dups['n_candidates'] == 1).sum()}")
print(f"  neither carries a balance:      {(dups['n_candidates'] == 0).sum()}  (identical rows; balance recovered in Act 5)")
print(f"\nOf the {len(both)} conflicting pairs:")
print(f"  a reconciling row exists in:            {both['kept_is_a_candidate'].sum()}")
print(f"  pairs where the losing row ALSO fits:   {(both['dropped_off_by'].abs() <= TOL).sum()}")
print(f"  losing row misses the ledger by:        min {both['dropped_off_by'].abs().min():,.1f} | median {both['dropped_off_by'].abs().median():,.0f} | max {both['dropped_off_by'].abs().max():,.0f} currency units (max {both['dropped_off_by_pct'].abs().max():.3f}% of the balance)")
print(f"  averaging instead would miss by half that gap (up to {both['mean_rule_error'].max():,.0f} units) and reconcile with neither neighbour")
both[["account_id", "date", "kept", "dropped", "dropped_off_by", "dropped_off_by_pct"]].head(8)

Duplicated (account, day) pairs: 48
  both rows carry a balance:      44
  only one row carries a balance: 0
  neither carries a balance:      4  (identical rows; balance recovered in Act 5)

Of the 44 conflicting pairs:
  a reconciling row exists in:            44
  pairs where the losing row ALSO fits:   0
  losing row misses the ledger by:        min 2.2 | median 27,823 | max 2,589,197 currency units (max 0.097% of the balance)
  averaging instead would miss by half that gap (up to 1,294,599 units) and reconcile with neither neighbour


,account_id,date,kept,dropped,dropped_off_by,dropped_off_by_pct
42,ACC-001,2025-02-12,"2,369,696,014.90","2,369,590,098.91","-105,915.99",-0.00
69,ACC-001,2025-03-11,"2,584,816,054.79","2,585,833,800.52","1,017,745.73",0.04
135,ACC-001,2025-05-16,"2,470,396,939.75","2,472,400,664.12","2,003,724.37",0.08
138,ACC-001,2025-05-19,"2,416,702,188.16","2,416,744,336.52","42,148.36",0.00
177,ACC-001,2025-06-27,"2,847,532,254.14","2,848,352,103.93","819,849.79",0.03
199,ACC-001,2025-07-19,"2,681,694,987.91","2,682,811,747.50","1,116,759.59",0.04
228,ACC-001,2025-08-17,"2,529,603,665.57","2,531,207,954.64","1,604,289.07",0.06
283,ACC-002,2025-01-14,"174,680.22","174,713.96",33.74,0.02


In **every** conflicting pair exactly one row reconciles with its neighbours to the cent, and the
other misses by a small-but-real amount (≤ 0.1% of the balance) — never zero. That's why averaging
is wrong: the mean lands *between* a correct value and a wrong one, so it matches the ledger on
neither side.

**Why it matters.** A trailing-window calculation over unresolved duplicates picks up whichever
row sorts last — arbitrarily. The error is tiny per row, but it's systematic noise that a
threshold rule can't distinguish from real movement.

**Resolved:** keep the reconciling row; where both rows are blank the balance is recovered by the
identity (Act 5). The likely root cause — a re-load that rewrote the balance but not the flows —
is worth confirming at the source (open item).

## Act 5 — Missing values: the identity fills them exactly

**What we saw.** About 4% of `balance`, `inflow` and `outflow` are blank, spread across every
account (no account or window absorbs them). The usual options — drop, forward-fill,
interpolate — would all *guess*. Here we don't have to: with two of the three numbers on a row
(plus the previous balance) the third is determined.

In [10]:
raw_missing = raw[["balance", "inflow", "outflow"]].isna().sum().rename("blank_rows_in_raw")
by_account = raw.loc[raw["balance"].isna(), "account_id"].value_counts().sort_index()
print("Blank balances by account:", by_account.to_dict())

day_level = pd.DataFrame({
    "blank_days_in_raw": [
        int((ledger["n_candidates"] == 0).sum()),
        int(ledger["inflow_raw"].isna().sum()),
        int(ledger["outflow_raw"].isna().sum()),
    ],
    "recovered_exactly": [
        int(ledger["balance_status"].str.startswith("imputed").sum()),
        int((ledger["flow_status"] == "inflow_recovered").sum()),
        int((ledger["flow_status"] == "outflow_recovered").sum()),
    ],
}, index=["balance", "inflow", "outflow"])

quarantined = ledger["balance_status"].eq("unreconciled_episode")
day_level["still_missing"] = [
    int(ledger["balance_clean"].isna().sum()),
    int(ledger["inflow_clean"].isna().sum()),
    int(ledger["outflow_clean"].isna().sum()),
]
day_level.insert(0, "blank_rows_in_raw_file", raw_missing.to_numpy())
day_level

Blank balances by account: {'ACC-001': 11, 'ACC-002': 9, 'ACC-003': 16, 'ACC-004': 12, 'ACC-005': 10, 'ACC-006': 10}


,blank_rows_in_raw_file,blank_days_in_raw,recovered_exactly,still_missing
balance,68,64,64,18
inflow,65,64,60,4
outflow,65,64,58,6


Almost everything blank is recovered. What's left: the balances inside the quarantined episodes
(Act 6 — blank *on purpose*) and a handful of flows that no equation can reach — days where both
flows are missing at once (one equation, two unknowns), the very first day of a series (no
previous balance), and one day inside a quarantined window.

**Is "exact" really exact?** Trust, but verify: hide 10% of the balances we *do* trust, rerun the
whole procedure blind, and compare with what was actually reported.

In [11]:
truth = ledger[ledger["balance_status"] == "observed_reconciled"]
hidden = truth.sample(frac=0.10, random_state=42)[["account_id", "date", "balance_clean"]]

masked = raw.merge(hidden.rename(columns={"date": "date_parsed"})[["account_id", "date_parsed"]].assign(_hide=True),
                   on=["account_id", "date_parsed"], how="left")
masked.loc[masked["_hide"].notna(), "balance"] = np.nan

rebuilt = reconcile_ledger(masked.drop(columns="_hide"))
cmp = hidden.merge(rebuilt[["account_id", "date", "balance_clean", "balance_status"]], on=["account_id", "date"], suffixes=("_actual", "_rebuilt"))
cmp["abs_error"] = (cmp["balance_clean_rebuilt"] - cmp["balance_clean_actual"]).abs()
recovered = cmp["balance_clean_rebuilt"].notna()

print(f"Hidden balances: {len(cmp)}")
print(f"  rebuilt by the procedure: {recovered.sum()} ({recovered.mean():.1%})")
print(f"  max |error| among rebuilt:  {cmp.loc[recovered, 'abs_error'].max():.4f} currency units")
print(f"  rebuilt within 1 unit:      {(cmp.loc[recovered, 'abs_error'] <= 1).mean():.1%}")
print(f"  left blank (no safe way):   {(~recovered).sum()}")

Hidden balances: 152
  rebuilt by the procedure: 152 (100.0%)
  max |error| among rebuilt:  0.0100 currency units
  rebuilt within 1 unit:      100.0%
  left blank (no safe way):   0


The imputation is **not a model** — it's arithmetic. Hidden balances come back to the cent, and
where a value can't be rebuilt safely the procedure says so instead of guessing.

**Why it matters.** Interpolating a balance across a gap that contains a big outflow would
paper over exactly the days treasury cares about.

**Resolved:** balances and flows recovered by the identity, backtested; unrecoverable gaps stay
blank and flagged.

## Act 6 — Corrupted readings: sign flips, decimal shifts, and six 3-day episodes

**What we saw.** Four balances are negative, and a handful of one-day spikes swing 10× in either
direction. Are the accounts really going overdrawn? Let's ask the ledger.

### 6a — One-day errors with an exact signature

In [12]:
repaired = ledger[ledger["balance_status"].str.startswith("repaired")].copy()
repaired["reported"] = repaired["cand_1"]
repaired["ledger_says"] = repaired["balance_clean"]
repaired["reported / ledger"] = repaired["reported"] / repaired["ledger_says"]
repaired["error_type"] = repaired["balance_status"].str.replace("repaired_", "")
repaired = repaired.merge(accounts[["account_id", "currency", "account_type"]], on="account_id")
print(f"{len(repaired)} single-day corruptions, every one an exact multiple of the ledger value:")
repaired[["account_id", "account_type", "currency", "date", "reported", "ledger_says", "reported / ledger", "error_type"]].sort_values(["error_type", "account_id", "date"])

7 single-day corruptions, every one an exact multiple of the ledger value:


,account_id,account_type,currency,date,reported,ledger_says,reported / ledger,error_type
3,ACC-004,operational,MXN,2025-08-31,"201,223,848.30","20,122,384.83",10.00,decimal_shift_x10
5,ACC-005,reserve,MXN,2025-04-26,"1,569,232,059.50","156,923,205.96",10.00,decimal_shift_x10
6,ACC-006,reserve,USD,2025-05-03,"1,518,428.50","151,842.85",10.00,decimal_shift_x10
0,ACC-001,operational,COP,2025-07-18,"-2,615,645,702.74","2,615,645,702.74",-1.00,sign_flip
1,ACC-003,reserve,COP,2025-04-03,"-4,070,745,652.07","4,070,745,652.07",-1.00,sign_flip
2,ACC-003,reserve,COP,2025-06-24,"-2,890,859,520.02","2,890,859,520.01",-1.00,sign_flip
4,ACC-005,reserve,MXN,2025-04-10,"-161,888,673.61","161,888,673.61",-1.00,sign_flip


Each reading is *exactly* −1× or 10× what the ledger says it should be (the ratios above print as
−1.00 and 10.00) — a flipped sign and a slipped decimal, both reversed the next day. Real cash
that leaves an account doesn't come back the next morning without a matching inflow, and the
recorded flows on those days don't call for any such jump. These are recording errors, and
they're repaired exactly.

**Correction to an earlier finding.** In the first-pass diagnostics we read the four negative
balances as evidence that the reserve accounts really went overdrawn while the squad's rule wasn't
watching them. The ledger says otherwise:

In [13]:
neg = ledger[ledger["cand_1"] < 0][["account_id", "date", "cand_1", "balance_clean", "balance_status"]]
neg = neg.rename(columns={"cand_1": "reported", "balance_clean": "ledger_says"})
print(f"Negative readings in the raw data: {len(neg)}  → negative balances after reconciliation: {int((ledger['balance_clean'] < 0).sum())}")
neg

Negative readings in the raw data: 4  → negative balances after reconciliation: 0


,account_id,date,reported,ledger_says,balance_status
198,ACC-001,2025-07-18,"-2,615,645,702.74","2,615,645,702.74",repaired_sign_flip
632,ACC-003,2025-04-03,"-4,070,745,652.07","4,070,745,652.07",repaired_sign_flip
714,ACC-003,2025-06-24,"-2,890,859,520.02","2,890,859,520.01",repaired_sign_flip
1179,ACC-005,2025-04-10,"-161,888,673.61","161,888,673.61",repaired_sign_flip


All four negatives are sign flips; after repair **no account is ever negative**. So those rows are
data errors, not shortfalls. (The squad's `THRESHOLD` dict still has no entry for the two reserve
accounts — that's a real design gap — but this data doesn't show it costing anything.)

### 6b — Three-day drops that reconcile with nothing

The 1,586 remaining day-to-day steps all reconcile. What's left over is not a scattering of
errors but **six identical-looking episodes**: the balance falls 65–85% for three days, then
snaps back. Unlike sign flips, they have no exact signature and the flows don't explain them.

In [14]:
ep = ledger[ledger["balance_status"] == "unreconciled_episode"].sort_values(["account_id", "date"]).copy()
ep["episode"] = (ep.groupby("account_id")["date"].diff().dt.days != 1).cumsum()

rows = []
for _, g in ep.groupby("episode"):
    acc, start, end = g["account_id"].iloc[0], g["date"].min(), g["date"].max()
    acct = ledger[ledger["account_id"] == acc].set_index("date")
    before = acct.loc[start - pd.Timedelta(days=1), "balance_clean"]
    after_day = end + pd.Timedelta(days=1)
    after = acct.loc[after_day, "balance_clean"]
    window = acct.loc[start:after_day]
    flows_known = window[["inflow_clean", "outflow_clean"]].notna().all().all()
    ledger_path = before + (window["inflow_clean"] - window["outflow_clean"]).sum() if flows_known else np.nan
    cur = accounts.set_index("account_id").loc[acc, "currency"]
    jump = before - g["cand_1"].iloc[0]
    near = transfers[
        (transfers["currency"] == cur)
        & ((transfers["from_account"] == acc) | (transfers["to_account"] == acc))
        & (transfers["date_settled"].between(start - pd.Timedelta(days=3), after_day + pd.Timedelta(days=3)))
        & (transfers["amount"] >= 0.5 * jump)
    ]
    rows.append({
        "account_id": acc, "start": start.date(), "end": end.date(), "days": len(g),
        "last_good_balance": before, "lowest_reported": g["cand_1"].min(),
        "drop_pct": (g["cand_1"].min() / before - 1) * 100,
        "balance_after": after, "ledger_would_say": ledger_path,
        "unexplained_by_flows": after - ledger_path if flows_known else np.nan,
        "unexplained_pct_of_last_good": (after - ledger_path) / before * 100 if flows_known else np.nan,
        "squad_threshold": SQUAD_THRESHOLD.get(acc, np.nan),
        "transfers_of_similar_size_nearby": len(near),
        "largest_transfer_in_currency": transfers.loc[transfers["currency"] == cur, "amount"].max(),
    })
episodes = pd.DataFrame(rows)

flow_lookup = ledger.set_index(["account_id", "date"])
matched = 0
for _, tr in transfers.iterrows():
    hit = False
    for d in (tr["date_requested"], tr["date_settled"]):
        for acc, col in ((tr["from_account"], "outflow_raw"), (tr["to_account"], "inflow_raw")):
            if (acc, d) in flow_lookup.index:
                v = flow_lookup.loc[(acc, d), col]
                hit |= bool(pd.notna(v) and abs(v - tr["amount"]) <= 1.0)
    matched += hit
print(f"Transfers whose amount matches a daily inflow/outflow of the accounts involved: {matched} of {len(transfers)}")
print(f"Exact duplicate rows in the transfer log (identical except transfer_id): "
      f"{int(transfers.duplicated(subset=[c for c in transfers.columns if c != 'transfer_id'], keep=False).sum())}")
episodes

Transfers whose amount matches a daily inflow/outflow of the accounts involved: 0 of 145
Exact duplicate rows in the transfer log (identical except transfer_id): 10


,account_id,start,end,days,last_good_balance,lowest_reported,drop_pct,balance_after,ledger_would_say,unexplained_by_flows,unexplained_pct_of_last_good,squad_threshold,transfers_of_similar_size_nearby,largest_transfer_in_currency
0,ACC-001,2025-03-18,2025-03-20,3,"2,619,115,895.18","646,843,726.03",-75.30,"2,424,256,729.37","1,713,790,149.55","710,466,579.82",27.13,1000000000,0,"296,223,185.47"
1,ACC-001,2025-08-06,2025-08-08,3,"2,372,223,607.46","817,997,060.98",-65.52,"2,380,998,110.34","2,161,666,880.22","219,331,230.12",9.25,1000000000,0,"296,223,185.47"
2,ACC-002,2025-03-21,2025-03-23,3,"188,066.80","42,791.71",-77.25,"194,893.21","178,113.58","16,779.63",8.92,100000,0,"116,558.25"
3,ACC-002,2025-08-18,2025-08-20,3,"114,360.43","17,265.10",-84.90,"110,242.83",NaN,NaN,NaN,100000,4,"116,558.25"
4,ACC-004,2025-05-03,2025-05-05,3,"34,721,251.89","8,341,061.87",-75.98,"34,553,648.40","28,754,003.87","5,799,644.53",16.70,40000000,0,"115,805.54"
5,ACC-004,2025-09-14,2025-09-16,3,"20,351,369.25","4,290,099.41",-78.92,"18,628,916.32","15,826,359.43","2,802,556.89",13.77,40000000,0,"115,805.54"


What the table says:

- **The flows can't explain them.** Rolling the last good balance forward through the recorded
  flows, the balance re-appears 9–27% away from where the flows say it should be
  (`unexplained_pct_of_last_good`) — against a tolerance of one currency unit everywhere else.
  Entry *and* exit are unexplained, so the flows recorded inside the window are suspect too. (One
  window has a blank flow, so no bridge can be computed for it.)
- **The transfer log can't explain them either.** In five of six episodes no transfer comes close
  to the size of the jump. In the sixth (ACC-002, August) there are outgoing USD transfers of
  similar size — but those would push the balance *down*, whereas the unexplained event is the
  *recovery*. More fundamentally, **none of the 145 transfers appears as a matching amount in the
  daily flows** (0 of 145, printed above), so the log can't be used to reconcile balances at all.
- **Every episode dips below the squad's own threshold** for the account — which is why they
  matter: if real, they are the shortfalls the rule exists to catch; if artifacts, false alarms.
  (ACC-004 was already below its 40M threshold before both of its episodes.)

The data cannot tell us which. So we do the honest thing: **quarantine** — set these 18 balances
(and the flows recorded inside the windows) to missing, keep the raw values in the file, flag
them, and put the question to the owner of the source system. We do *not* repair them, because
there is no exact signature to repair with, and a plausible-looking guess is the most dangerous
thing to hand a treasurer.

**Why it matters.** Note what the squad's 14-day trailing mean does to a 3-day, ~75% drop:
it dilutes it to a ~16% dip, far above the threshold — so if these are real shortfalls, the rule
is blind to them; and if they're errors, they contaminate 14 forecast days each (Act 10).
Either way the current approach is wrong; which way is an open question for the source.

**Resolved:** 7 single-day errors repaired exactly; 6 episodes quarantined with evidence.

## Act 7 — The clean dataset

We assemble one tidy table: cleaned `balance`/`inflow`/`outflow`, the account's catalogue
currency, and a flag saying how each balance earned its place. Flows recorded inside quarantined
windows are blanked with the balances they belong to.

In [15]:
clean = ledger.merge(accounts[["account_id", "currency", "account_type", "bank_name"]], on="account_id")
in_episode = clean["balance_status"] == "unreconciled_episode"
clean["balance"] = clean["balance_clean"]
clean["inflow"] = clean["inflow_clean"].where(~in_episode)
clean["outflow"] = clean["outflow_clean"].where(~in_episode)
clean = clean.sort_values(["account_id", "date"]).reset_index(drop=True)

# Audit: does every pair of consecutive clean days obey the identity?
audit = clean.copy()
audit["prev_balance"] = audit.groupby("account_id")["balance"].shift(1)
audit["residual"] = audit["balance"] - (audit["prev_balance"] + audit["inflow"] - audit["outflow"])
ev = audit.dropna(subset=["residual"])
print(f"AUDIT  day-pairs testable: {len(ev):,} | reconcile within 1 unit: {(ev['residual'].abs() <= 1).sum():,} "
      f"({(ev['residual'].abs() <= 1).mean():.1%}) | worst residual: {ev['residual'].abs().max():.4f}")
print(f"       negative balances: {int((clean['balance'] < 0).sum())} | "
      f"blank balances (quarantined): {int(clean['balance'].isna().sum())} | "
      f"account-days: {len(clean):,}")

out_cols = ["account_id", "date", "currency", "account_type", "balance", "inflow", "outflow", "balance_status", "flow_status"]
clean[out_cols].to_csv("account_balances_daily_CLEAN.csv", index=False)
print("saved → account_balances_daily_CLEAN.csv")
clean[out_cols].head()

AUDIT  day-pairs testable: 1,586 | reconcile within 1 unit: 1,586 (100.0%) | worst residual: 0.0100
       negative balances: 0 | blank balances (quarantined): 18 | account-days: 1,620
saved → account_balances_daily_CLEAN.csv


,account_id,date,currency,account_type,balance,inflow,outflow,balance_status,flow_status
0,ACC-001,2025-01-01,COP,operational,"2,579,043,824.12","94,044,817.48","15,000,993.36",observed_reconciled,
1,ACC-001,2025-01-02,COP,operational,"2,561,627,958.95","125,757,477.77","143,173,342.95",observed_reconciled,
2,ACC-001,2025-01-03,COP,operational,"2,561,627,958.95",0.00,0.00,observed_reconciled,
3,ACC-001,2025-01-04,COP,operational,"2,588,018,042.02","38,926,524.28","12,536,441.22",observed_reconciled,
4,ACC-001,2025-01-05,COP,operational,"2,617,987,216.05","29,969,174.04",0.00,observed_reconciled,


Before/after, for every account. Left: the date-fixed raw balances, nothing else touched. Right:
the reconciled series, with repaired days (◆), exactly-imputed days (○) and quarantined windows
(shaded). *Double-click a panel to reset the zoom.*

In [16]:
accs = sorted(clean["account_id"].unique())
fig = make_subplots(
    rows=len(accs), cols=2, horizontal_spacing=0.08, vertical_spacing=0.035,
    subplot_titles=[t for a in accs for t in (f"{a} — raw", f"{a} — clean")],
)
for r, acc in enumerate(accs, start=1):
    g = clean[clean["account_id"] == acc]
    color = ACCOUNT_COLORS[acc]
    first = r == 1
    fig.add_trace(go.Scatter(x=g["date"], y=g["cand_1"], mode="lines", line=dict(color="#9AA0A6", width=1.2),
                             name="Raw (dates fixed only)", showlegend=first, legendgroup="raw"), row=r, col=1)
    fig.add_trace(go.Scatter(x=g["date"], y=g["balance"], mode="lines", line=dict(color=color, width=1.4),
                             name="Clean", showlegend=False), row=r, col=2)
    rep = g[g["balance_status"].str.startswith("repaired")]
    fig.add_trace(go.Scatter(x=rep["date"], y=rep["balance"], mode="markers", name="Repaired (exact)",
                             marker=dict(symbol="diamond", size=9, color="#E8A33D", line=dict(color="black", width=0.8)),
                             showlegend=first, legendgroup="rep"), row=r, col=2)
    imp = g[g["balance_status"].str.startswith("imputed")]
    fig.add_trace(go.Scatter(x=imp["date"], y=imp["balance"], mode="markers", name="Imputed via ledger",
                             marker=dict(symbol="circle-open", size=6, color="#333333"),
                             showlegend=first, legendgroup="imp"), row=r, col=2)
    for _, e in episodes[episodes["account_id"] == acc].iterrows():
        x0 = (pd.Timestamp(e["start"]) - pd.Timedelta(hours=12)).isoformat()
        x1 = (pd.Timestamp(e["end"]) + pd.Timedelta(hours=12)).isoformat()
        for c in (1, 2):
            fig.add_vrect(x0=x0, x1=x1, row=r, col=c, fillcolor="#C44E52", opacity=0.18, line_width=0)

fig.add_trace(go.Scatter(x=[None], y=[None], mode="markers", name="Quarantined 3-day episode",
                         marker=dict(symbol="square", size=11, color="rgba(196,78,82,0.35)")), row=1, col=2)
fig.update_layout(height=250 * len(accs), template="plotly_white", hovermode="x unified",
                  title="Balance by account — before vs after reconciliation",
                  margin=dict(t=140), legend=dict(orientation="h", y=1.055, x=0))
fig.show()

The left column is what any model trained on the raw file would learn from — 10× spikes, negative
troughs, and quiet 3-day collapses. The right column is a series in which every point either
reconciles with the flows or is explicitly flagged as not doing so.

**Resolved & saved:** `account_balances_daily_CLEAN.csv` (1,620 account-days), ready for Part 2.

## Act 8 — Outliers revisited: proven errors vs. real variability

Earlier, a robust z-score flagged 59 "outliers". A statistical flag says *unusual*; it can't say
*wrong*. Now we can tell the two apart: a flagged point is **explained** if the ledger already
proved it a corrupted reading (repaired or quarantined), and **legitimate** otherwise.

In [17]:
def mod_zscore(s):
    med = s.median()
    mad = (s - med).abs().median()
    if not mad or pd.isna(mad):
        return pd.Series(0.0, index=s.index)
    return 0.6745 * (s - med) / mad

Z = 3.5


def flag_outliers(df, date_col):
    d = df.sort_values(["account_id", date_col]).copy()
    d["balance_change"] = d.groupby("account_id")["balance"].diff()
    z = {
        "balance": d.groupby("account_id")["balance_change"].transform(mod_zscore),   # change, so trends aren't "outliers"
        "inflow": d.groupby("account_id")["inflow"].transform(mod_zscore),
        "outflow": d.groupby("account_id")["outflow"].transform(mod_zscore),
    }
    parts = [d.loc[z[s].abs() > Z, ["account_id", date_col]].rename(columns={date_col: "date"}).assign(series=s) for s in z]
    return pd.concat(parts, ignore_index=True)


flags_raw = flag_outliers(naive_daily, "date_parsed")
flags_clean = flag_outliers(clean.rename(columns={"date": "d"}), "d")

status = ledger.set_index(["account_id", "date"])["balance_status"]
is_error = lambda a, d: status.get((a, d), "").startswith("repaired") or status.get((a, d), "") == "unreconciled_episode"


def explained(r):
    if r["series"] == "balance":
        return is_error(r["account_id"], r["date"]) or is_error(r["account_id"], r["date"] - pd.Timedelta(days=1))
    return status.get((r["account_id"], r["date"]), "") == "unreconciled_episode"


flags_raw["explained_by_proven_error"] = flags_raw.apply(explained, axis=1)
summary = (
    flags_raw.groupby("series")["explained_by_proven_error"].agg(flagged="size", explained="sum")
    .assign(legitimate=lambda d: d["flagged"] - d["explained"])
    .join(flags_clean.groupby("series").size().rename("still_flagged_on_clean_data"))
    .fillna(0).astype(int)
)
summary.loc["TOTAL"] = summary.sum()
summary

,flagged,explained,legitimate,still_flagged_on_clean_data
series,,,,
balance,36,24,12,11
inflow,8,0,8,9
outflow,15,3,12,15
TOTAL,59,27,32,35


Two thirds of the *balance* flags (24 of 36) were proven errors — sign flips, decimal shifts and
episodes — and the reconciled series no longer contains them. The *flow* flags are mostly legitimate: large but
ledger-consistent inflows and outflows (payment runs, top-ups). Those are real variability and
must stay in the data — deleting them would erase exactly the events a liquidity model has to
handle.

**Why it matters.** "Remove outliers" is a common reflex that would have destroyed real large
payments while leaving the subtle errors. Proving *why* a point is wrong is what makes it safe
to touch.

## Act 9 — What structure is left? Seasonality on the clean series

With the corruption gone, does the data have a rhythm worth forecasting? MSTL decomposes each
series into trend + a weekly (7-day) + a ~30-day component + residual (`robust=True`). It needs a
complete series, so the few remaining blanks are linearly interpolated **for this analysis only**.
*Strength* = share of the non-trend variation a component explains (0 = none, 1 = perfectly periodic).
The 30-day cycle is counted from 2025-01-01, not aligned to calendar months.

In [18]:
from statsmodels.tsa.seasonal import MSTL


def strength(component, resid):
    return max(0.0, 1 - np.var(resid) / np.var(component + resid))


acc_type = accounts.set_index("account_id")["account_type"]
rows, profile = [], {}
for acc, g in clean.groupby("account_id"):
    g = g.set_index("date").asfreq("D")
    for col in ["balance", "inflow", "outflow"]:
        s = g[col].interpolate(limit_direction="both")
        res = MSTL(s, periods=[7, 30], stl_kwargs={"robust": True}).fit()
        rows.append({"account_id": acc, "type": acc_type[acc], "series": col,
                     "weekly": strength(res.seasonal["seasonal_7"], res.resid),
                     "monthly": strength(res.seasonal["seasonal_30"], res.resid)})
        if col != "balance":
            prof = res.seasonal["seasonal_7"].groupby(res.seasonal.index.dayofweek).mean() / s.mean() * 100
            profile[f"{acc} ({acc_type[acc]}) {col}"] = prof.to_numpy()

strengths = pd.DataFrame(rows)
strengths.pivot(index=["account_id", "type"], columns="series", values=["weekly", "monthly"]).round(2)

weekly                monthly               
series                 balance inflow outflow balance inflow outflow
account_id type                                                     
ACC-001    operational    0.01   0.11    0.19    0.00   0.00    0.13
ACC-002    operational    0.04   0.12    0.21    0.05   0.00    0.14
ACC-003    reserve        0.02   0.54    0.57    0.07   0.08    0.16
ACC-004    operational    0.01   0.24    0.04    0.32   0.20    0.26
ACC-005    reserve        0.04   0.40    0.50    0.18   0.08    0.00
ACC-006    reserve        0.06   0.38    0.33    0.00   0.07    0.04

In [19]:
days = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
labels = list(profile)
z = np.array([profile[k] for k in labels])
fig = go.Figure(go.Heatmap(
    z=z, x=days, y=labels, colorscale="RdBu", zmid=0, zmin=-100, zmax=100,
    text=np.round(z).astype(int), texttemplate="%{text}", colorbar=dict(title="% of avg<br>daily flow"),
    hovertemplate="%{y}<br>%{x}: %{z:.0f}% of the average day<extra></extra>",
))
fig.update_layout(template="plotly_white", height=460, title="Weekday effect on daily flows (weekly component, % of the series' average day)",
                  yaxis=dict(autorange="reversed"))
fig.show()

Reading it:

- **`balance` carries no weekly rhythm** (strength ≤ 0.06 for every account) — what we'd expect
  from a stock, which accumulates whatever the flows did rather than resetting. The ~30-day
  figures for ACC-004 (0.32) and ACC-005 (0.18) balances deserve caution: both decline steadily,
  and a trend can leak into a long-period component.
- **The flows do.** Weekday/weekend structure is strong in the reserve accounts (ACC-003,
  ACC-005: roughly 0.4–0.6) and present in ACC-006, with weekdays above average and weekends
  sharply below — banking activity that stops on weekends. The operational accounts show the same
  weekend dip in the heatmap but with much more day-to-day noise, hence their lower strength.
- **The ~30-day rhythm is weaker and messier**, but stands out for **ACC-004** — the account the
  squad's notebook called unreliable — in `inflow` (0.20) and `outflow` (0.26). It survives
  cleaning, so it isn't an artifact of the corrupted rows.

**Why it matters.** A flat 14-day mean treats Saturday like Wednesday. In 10 of the 12 flow series
both weekend days sit roughly 25–60% below the average day (heatmap above) — that's not noise
being smoothed, it's a predictable swing being ignored. This is where Part 2 should start: forecast the flows, not the
balance level.

## Act 10 — So what? Does any of this change the squad's answer?

We re-run the squad's own forecast — mean of the last 14 balances — on their pipeline and on the
clean series, for every as-of date, without changing their logic. That isolates the effect of the
*data* from the effect of the *method*.

In [20]:
squad = pd.read_csv("account_balances_daily_RAW.csv")
squad["date"] = pd.to_datetime(squad["date"], errors="coerce")
squad = squad.dropna(subset=["date"]).sort_values(["account_id", "date"])

asofs = pd.date_range("2025-01-14", clean["date"].max())
rows = []
for acc in sorted(clean["account_id"].unique()):
    s_squad = squad[squad["account_id"] == acc]
    s_clean = clean[clean["account_id"] == acc].set_index("date")["balance"]
    for d in asofs:
        rows.append((acc, d, s_squad[s_squad["date"] <= d]["balance"].tail(14).mean(), s_clean[:d].tail(14).mean()))
fc = pd.DataFrame(rows, columns=["account_id", "asof", "squad_forecast", "clean_forecast"])
fc["deviation"] = fc["squad_forecast"] / fc["clean_forecast"] - 1

def rule_fires(acc, series):
    t = SQUAD_THRESHOLD.get(acc)
    return int((series < t).sum()) if t else np.nan

impact = fc.groupby("account_id").agg(
    max_abs_deviation=("deviation", lambda x: x.abs().max()),
    days_off_by_over_10pct=("deviation", lambda x: int((x.abs() > 0.10).sum())),
    as_of_days=("deviation", "size"),
)
impact["rule_fires_squad"] = [rule_fires(a, fc.loc[fc["account_id"] == a, "squad_forecast"]) for a in impact.index]
impact["rule_fires_clean"] = [rule_fires(a, fc.loc[fc["account_id"] == a, "clean_forecast"]) for a in impact.index]
impact["squad_threshold"] = [SQUAD_THRESHOLD.get(a, np.nan) for a in impact.index]
impact.style.format({"max_abs_deviation": "{:.0%}", "squad_threshold": "{:,.0f}",
                     "rule_fires_squad": "{:.0f}", "rule_fires_clean": "{:.0f}"}, na_rep="—")

,max_abs_deviation,days_off_by_over_10pct,as_of_days,rule_fires_squad,rule_fires_clean,squad_threshold
account_id,,,,,,
ACC-001,17%,36,257,0,0,"1,000,000,000"
ACC-002,21%,30,257,26,14,"100,000"
ACC-003,15%,17,257,—,—,—
ACC-004,69%,46,257,187,188,"40,000,000"
ACC-005,68%,32,257,—,—,—
ACC-006,69%,18,257,0,0,"100,000"


In [21]:
last = fc[fc["asof"] == fc["asof"].max()].set_index("account_id")[["squad_forecast", "clean_forecast", "deviation"]]
fires = lambda a, f: (SQUAD_THRESHOLD[a] > f) if a in SQUAD_THRESHOLD else "no threshold"
last["squad_rule_fires"] = [fires(a, last.loc[a, "squad_forecast"]) for a in last.index]
last["clean_rule_fires"] = [fires(a, last.loc[a, "clean_forecast"]) for a in last.index]
print(f"Final as-of date: {fc['asof'].max().date()} — the numbers the squad's notebook actually reports")
last.style.format({"squad_forecast": "{:,.0f}", "clean_forecast": "{:,.0f}", "deviation": "{:+.1%}"})

Final as-of date: 2025-09-27 — the numbers the squad's notebook actually reports


,squad_forecast,clean_forecast,deviation,squad_rule_fires,clean_rule_fires
account_id,,,,,
ACC-001,"2,305,518,572","2,309,307,897",-0.2%,False,False
ACC-002,"79,790","78,886",+1.1%,True,True
ACC-003,"2,920,535,273","2,938,621,383",-0.6%,no threshold,no threshold
ACC-004,"15,965,982","18,955,526",-15.8%,True,True
ACC-005,"107,312,810","106,983,524",+0.3%,no threshold,no threshold
ACC-006,"151,186","151,057",+0.1%,False,False


In [22]:
accs = sorted(clean["account_id"].unique())
fig = make_subplots(rows=3, cols=2, vertical_spacing=0.09, horizontal_spacing=0.08,
                    subplot_titles=[f"{a} ({acc_type[a]})" for a in accs])
for i, acc in enumerate(accs):
    r, c = i // 2 + 1, i % 2 + 1
    g = fc[fc["account_id"] == acc]
    fig.add_trace(go.Scatter(x=g["asof"], y=g["squad_forecast"], mode="lines", line=dict(color="#9AA0A6", width=1.6),
                             name="Squad pipeline", showlegend=i == 0, legendgroup="s"), row=r, col=c)
    fig.add_trace(go.Scatter(x=g["asof"], y=g["clean_forecast"], mode="lines", line=dict(color="#2B6CB0", width=1.8),
                             name="Same method, clean data", showlegend=False), row=r, col=c)
    if acc in SQUAD_THRESHOLD:
        fig.add_hline(y=SQUAD_THRESHOLD[acc], line=dict(color="#C44E52", dash="dash", width=1), row=r, col=c)
fig.add_trace(go.Scatter(x=[None], y=[None], mode="lines", line=dict(color="#2B6CB0", width=1.8), name="Same method, clean data"))
fig.add_trace(go.Scatter(x=[None], y=[None], mode="lines", line=dict(color="#C44E52", dash="dash"), name="Squad threshold"))
fig.update_layout(height=820, template="plotly_white", hovermode="x unified", legend=dict(orientation="h", y=1.07, x=0),
                  title="The squad's 14-day forecast, by as-of date: their pipeline vs the same method on clean data")
fig.show()

Two honest findings — one against the data, one *for* it:

1. **The data problems corrupt the forecast series, often and for a long time.** Each bad reading
   sits in the 14-day window for two weeks: one 10× spike lifts the mean by 9/14 ≈ 64% (ACC-004,
   -005, -006), and a sign flip or an episode drags it down. Across as-of dates the squad's
   forecast is more than 10% off the clean-data forecast on roughly 7–18% of days per account, and
   up to ~69% off at the worst. On ACC-002 the rule fires on 26 as-of days with the squad's
   pipeline versus 14 on clean data — 12 more alert days than the data justifies. That is what
   "noisy and unreliable" looked like from the outside.
2. **But fixing the data alone would not have changed the final-day recommendations.** On the
   date the squad reports, only ACC-004 moves materially (≈ −16%), and the two transfers they
   recommend (ACC-002, ACC-004) fire on clean data too. ACC-004's threshold (40M) sits above the
   account's *median* balance (≈31M), so the rule fires on ~73% of all days regardless — a
   structural mis-calibration, not noise. And the donor rule still compares raw balances across
   COP, USD and MXN, so it always picks a COP account.

That's the real lesson of Part 1: **clean data is necessary but not sufficient.** The remaining
flaws — fixed thresholds, missing coverage for the reserve accounts, cross-currency donor
choice, and a before/after comparison that averages balances across currencies — are modelling
problems, and they're Part 2's job. What Part 1 delivers is the ground to stand on.

## Scorecard, open items and what comes next

In [23]:
n_status = ledger["balance_status"].value_counts()
scorecard = pd.DataFrame([
    ("Dates",      f"errors='coerce' + dropna drops {dropped} of {len(raw):,} rows ({dropped/len(raw):.1%})",
                   "Separator rule: '/' = day-first, '-' = month-first (0 exceptions)",
                   f"{raw['date_parsed'].notna().sum():,}/{len(raw):,} rows recovered; 1 row per account per calendar day"),
    ("Currency",   f"{raw['currency_label'].nunique()} spellings for {raw['currency_norm'].nunique()} currencies",
                   "Assign accounts.csv currency; no FX", "0 rows contradict the catalogue"),
    ("Duplicates", f"{len(dups)} duplicated (account, day) pairs, {len(both)} with conflicting balances",
                   "Keep the row that satisfies the ledger identity",
                   f"{int(both['kept_is_a_candidate'].sum())}/{len(both)} conflicts decided by the ledger; mean rule would be off in all"),
    ("Missing",    f"{int(day_level.loc['balance','blank_days_in_raw'])} balance / {int(day_level.loc['inflow','blank_days_in_raw'])} inflow / {int(day_level.loc['outflow','blank_days_in_raw'])} outflow blank days",
                   "Solve for the unknown with balance_t = balance_(t-1) + inflow − outflow",
                   f"Backtest: {recovered.mean():.0%} of hidden balances rebuilt, max error {cmp.loc[recovered, 'abs_error'].max():.2f} units"),
    ("Sign flips / x10", f"{len(repaired)} one-day corruptions, incl. all 4 negative balances",
                   "Replace with the ledger value (exact −1x / 10x signature)", "Negative balances after repair: 0"),
    ("3-day episodes", f"{len(episodes)} windows, {int(n_status.get('unreconciled_episode', 0))} account-days, balance −65 to −85%",
                   "Quarantine (blank + flag); not repaired", "Neither flows nor transfers explain them → source must confirm"),
    ("Whole dataset", "—", "account_balances_daily_CLEAN.csv",
                   f"{(ev['residual'].abs() <= 1).mean():.0%} of {len(ev):,} testable day-pairs reconcile (worst {ev['residual'].abs().max():.2f})"),
], columns=["Issue", "What was wrong", "Resolution", "Proof"])
scorecard.style.hide(axis="index").set_properties(**{"text-align": "left", "white-space": "normal"})

Issue,What was wrong,Resolution,Proof
Dates,"errors='coerce' + dropna drops 293 of 1,668 rows (17.6%)","Separator rule: '/' = day-first, '-' = month-first (0 exceptions)","1,668/1,668 rows recovered; 1 row per account per calendar day"
Currency,9 spellings for 3 currencies,Assign accounts.csv currency; no FX,0 rows contradict the catalogue
Duplicates,"48 duplicated (account, day) pairs, 44 with conflicting balances",Keep the row that satisfies the ledger identity,44/44 conflicts decided by the ledger; mean rule would be off in all
Missing,64 balance / 64 inflow / 64 outflow blank days,Solve for the unknown with balance_t = balance_(t-1) + inflow − outflow,"Backtest: 100% of hidden balances rebuilt, max error 0.01 units"
Sign flips / x10,"7 one-day corruptions, incl. all 4 negative balances",Replace with the ledger value (exact −1x / 10x signature),Negative balances after repair: 0
3-day episodes,"6 windows, 18 account-days, balance −65 to −85%",Quarantine (blank + flag); not repaired,Neither flows nor transfers explain them → source must confirm
Whole dataset,—,account_balances_daily_CLEAN.csv,"100% of 1,586 testable day-pairs reconcile (worst 0.01)"


### Open items — decisions this notebook cannot make on its own

1. **The six 3-day episodes.** Real cash events or reporting artifacts? They are quarantined, not
   deleted. The source-system owner can settle it in one lookup, and the answer changes whether
   the squad's rule is *blind to real shortfalls* or *fed by false ones*.
2. **Why do duplicates exist?** In every conflicting pair only one balance ever reconciles, which
   points at a re-load that rewrote the balance but not the flows. Worth confirming and fixing at
   the source.
3. **11 `observed_unverified` balances and a few unrecoverable flows.** Kept and flagged because
   adjacent flows were missing; re-pulling those days from the bank would close them. (Flows were
   interpolated only for the seasonality analysis in Act 9.)
4. **Turn the ledger identity into a standing data check.** It caught every class of problem here
   and costs one line of SQL — a natural first quality gate for the production pipeline (Part 5).
5. **The transfer log doesn't reconcile with the balances.** None of its 145 transfers appears as
   a matching daily inflow/outflow, and 10 rows are exact duplicates (same everything but the
   id). Its role in the model (settlement lag? in-transit cash?) has to be defined before Part 2
   uses it.
6. **Not addressed here (model, not data):** threshold calibration and reserve-account coverage,
   the cross-currency donor rule, and the before/after comparison that averages balances across
   currencies — all Part 2.